In [4]:
PHASE_1_SYSTEM_PROMPT = """
You are an expert computational linguist and political scientist specializing in South Asian media and Bengali socio-political discourse.

Your task is to evaluate how a specific target identity is framed within a single provided Bengali news article. You will evaluate the narrative stance toward the target identity using a 5-point Likert scale:
+2: Highly Positive (Glorification, heroic portrayal, moral elevation)
+1: Positive (Supportive, approving portrayal)
0: Neutral (Factual, objective reporting without emotive cues)
-1: Negative (Critical, unfavorable portrayal, attribution of blame)
-2: Extremely Negative (Vilification, demonization, portrayal as a societal threat)

INSTRUCTIONS:
1. Analyze the lexical choices, cultural symbolism, and causal attribution regarding the specified Target Identity in the text. Pay close attention to Bengali diglossia and coded dogwhistles.
2. Draft a precise analytical reasoning chain (in English) explaining the narrative framing.
3. Provide the final framing score (-2 to +2).

OUTPUT FORMAT:
You must return a valid JSON object matching this schema:
{
  "comment": "Your precise English reasoning chain analyzing the linguistic framing",
  "rating": integer between -2 and 2
}
"""

In [ ]:
import json
import asyncio
import aiohttp
from typing import Dict

# Execution Configurations
OLLAMA_API_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2.5:7b"      # Multilingual open-source model running locally
CONCURRENT_REQUESTS = 4        # Scale based on local GPU VRAM limits

# --- FIX YOUR TARGET IDENTITY HERE ---
TARGET_IDENTITY = "ছায়ানট"       # Example: India (or "শেখ হাসিনা", "আওয়ামী লীগ", etc.)

async def screen_article_async(session: aiohttp.ClientSession, semaphore: asyncio.Semaphore, article: Dict) -> Dict:
    """
    Evaluates a single article against a fixed target identity using a local Ollama instance.
    """
    async with semaphore:
        # Dynamically inject the fixed identity and article context into the prompt
        user_prompt = f"Target Identity: {TARGET_IDENTITY}\n\nArticle Title: {article['title']}\nArticle Content:\n{article['content']}"
        full_prompt = f"{PHASE_1_SYSTEM_PROMPT}\n\n{user_prompt}"
        
        payload = {
            "model": MODEL_NAME,
            "prompt": full_prompt,
            "format": "json",         # Enforces strict JSON output to prevent markdown filler
            "stream": False,
            "options": {
                "num_ctx": 8192,      # Prevents truncation from Bengali token bloat
                "temperature": 0.1    # Enforces deterministic analytical output
            }
        }
        
        try:
            async with session.post(OLLAMA_API_URL, json=payload) as response:
                if response.status == 200:
                    data = await response.json()
                    raw_llm_response = data.get("response", "{}")
                    
                    try:
                        parsed_output = json.loads(raw_llm_response)
                        # Construct a lightweight record referencing key tracking elements
                        return {
                            "article_id": article["article_id"],
                            "title": article["title"],
                            # "cluster": article["cluster"],
                            "target_identity": TARGET_IDENTITY,
                            "rating": parsed_output.get("rating"),
                            "comment": parsed_output.get("comment"),
                            "status": "success"
                        }
                    except json.JSONDecodeError:
                        return {
                            "article_id": article["article_id"],
                            "title": article["title"],
                            # "cluster": article["cluster"],
                            "status": "schema_error"
                        }
                else:
                    return {"article_id": article["article_id"], "status": f"http_error_{response.status}"}
        except Exception as e:
            return {"article_id": article["article_id"], "status": f"exception_{str(e)}"}

async def run_phase_1(input_jsonl: str, output_jsonl: str):
    articles = []
    with open(input_jsonl, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                articles.append(json.loads(line))
                
    print(f"Loaded {len(articles)} articles. Starting screening for target: '{TARGET_IDENTITY}'...")
    
    semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)
    async with aiohttp.ClientSession() as session:
        tasks = [screen_article_async(session, semaphore, art) for art in articles]
        results = await asyncio.gather(*tasks)
        
    success_count = 0
    with open(output_jsonl, 'w', encoding='utf-8') as f:
        for res in results:
            f.write(json.dumps(res, ensure_ascii=False) + '\n')
            if res.get("status") == "success":
                success_count += 1
                
    print(f"Phase 1 Complete. Successfully screened {success_count}/{len(articles)} articles.")
# In a notebook cell
await run_phase_1(
    "llm_chayanot.jsonl",
    "qwen2.5_7b_chayanot.jsonl"
)

Loaded 12 articles. Starting screening for target: 'ছায়ানট'...
